# Week 4: Latent Feature Clustering

This notebook compares clustering using raw pixels, PCA features, and autoencoder latent features.

## 1. Setup and data

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                     if (folder / "week1" / "week1_ml_basics.ipynb").is_file()
                     and (folder / "week5" / "week5_vae.ipynb").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter in the research project folder or one of its subfolders.")
WEEK4_ROOT = PROJECT_ROOT / "week4"
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = WEEK4_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

transform = transforms.ToTensor()
full_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transform)
full_test = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transform)
rng = np.random.default_rng(seed)
train_indices = rng.choice(len(full_train), size=20000, replace=False)
test_indices = rng.choice(len(full_test), size=10000, replace=False)
train_loader = DataLoader(Subset(full_train, train_indices), batch_size=256, shuffle=True)
test_loader = DataLoader(Subset(full_test, test_indices), batch_size=256, shuffle=False)
X_images = np.stack([full_test[int(i)][0].numpy() for i in test_indices])
y_true = np.array([full_test[int(i)][1] for i in test_indices])
X_flat = X_images.reshape(len(X_images), -1)
print(X_flat.shape)


## Autoencoder model and training function

This notebook defines its own AE training function, so it can be run from the first cell without relying on another notebook.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid()
        )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.decoder(self.encode(x)).view(-1, 1, 28, 28)


def train_autoencoder(latent_dim, epochs=5):
    torch.manual_seed(seed)
    model = Autoencoder(latent_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    history = []
    for epoch in range(epochs):
        model.train()
        train_total = 0.0
        for images, _ in train_loader:
            images = images.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), images)
            loss.backward()
            optimizer.step()
            train_total += loss.item() * images.size(0)
        model.eval()
        test_total = 0.0
        with torch.inference_mode():
            for images, _ in test_loader:
                images = images.to(device)
                test_total += loss_fn(model(images), images).item() * images.size(0)
        history.append({
            "train_loss": train_total / len(train_loader.dataset),
            "test_loss": test_total / len(test_loader.dataset),
        })
        print(f"d={latent_dim}, epoch {epoch + 1}: test MSE={history[-1]['test_loss']:.4f}")
    return model, history


def encode_images(model, images):
    model.eval()
    batches = []
    with torch.inference_mode():
        for start in range(0, len(images), 256):
            batch = torch.from_numpy(images[start:start + 256]).to(device)
            batches.append(model.encode(batch).cpu().numpy())
    return np.concatenate(batches)


## 2. Define clustering evaluation

In [ ]:
def evaluate_kmeans(features, labels, name):
    cluster_ids = KMeans(n_clusters=10, n_init=10, random_state=seed).fit_predict(features)
    return {
        "Representation": name, "Dimension": features.shape[1],
        "NMI": normalized_mutual_info_score(labels, cluster_ids),
        "ARI": adjusted_rand_score(labels, cluster_ids),
    }


## 3. Compare raw pixels, PCA, and AE features

In [ ]:
Z_pca = PCA(n_components=20, random_state=seed).fit_transform(X_flat)
model_20, history_20 = train_autoencoder(20, epochs=5)
Z_ae = encode_images(model_20, X_images)
np.save(RESULTS_DIR / "ae_latent_20.npy", Z_ae)
np.save(RESULTS_DIR / "test_labels.npy", y_true)

rows = [
    evaluate_kmeans(X_flat, y_true, "Raw pixels"),
    evaluate_kmeans(Z_pca, y_true, "PCA-20"),
    evaluate_kmeans(Z_ae, y_true, "AE-20"),
]
comparison = pd.DataFrame(rows)
comparison.to_csv(RESULTS_DIR / "representation_comparison.csv", index=False)
comparison


## 4. Study latent dimensions 2, 10, 20, and 50

In [ ]:
dimension_rows = []
trained_models = {20: model_20}
for latent_dim in [2, 10, 20, 50]:
    if latent_dim == 20:
        model, history, latent = model_20, history_20, Z_ae
    else:
        model, history = train_autoencoder(latent_dim, epochs=5)
        latent = encode_images(model, X_images)
    trained_models[latent_dim] = model
    cluster_ids = KMeans(n_clusters=10, n_init=10, random_state=seed).fit_predict(latent)
    dimension_rows.append({
        "latent_dim": latent_dim,
        "reconstruction_loss": history[-1]["test_loss"],
        "NMI": normalized_mutual_info_score(y_true, cluster_ids),
        "ARI": adjusted_rand_score(y_true, cluster_ids),
    })

dimension_comparison = pd.DataFrame(dimension_rows)
dimension_comparison.to_csv(RESULTS_DIR / "ae_dimension_comparison.csv", index=False)
dimension_comparison


## 5. Visualize a 2D latent space

In [ ]:
# If the 2D model has been trained, color points by their ground-truth labels.
Z_2 = encode_images(trained_models[2], X_images)
plt.figure(figsize=(6, 5))
plt.scatter(Z_2[:, 0], Z_2[:, 1], c=y_true, s=3, cmap="tab10")
plt.xlabel("z1"); plt.ylabel("z2"); plt.title("AE latent space (dimension 2)"); plt.colorbar(); plt.show()


## Questions

- Does better reconstruction always mean better clustering?
- What happens when the latent dimension is very small?
- What happens when it is very large?

I will answer these questions using the tables and plots above.